# miRNA inference example 
## RCC scRNA-seq datasets - renal cell cancer (RCC). Accession GSE207493

Final model: stacking (Ridge): **TabPack (Muon) + XGB + TabM (AdamW)**.


In [12]:
path = '/home/amismailov/mirna-repo/ml_pipeline/final_train_test_inference/inference/target_config.json'
import json

with open(path, "r") as f:
    config = json.load(f)

In [13]:
ctr = 0
for i in config['cohorts'].keys():
    ctr += len(config['cohorts'][i])
print(ctr)

152


In [1]:
import sys
import json
from pathlib import Path
import pandas as pd

from preprocessor import SingleCell
from constants import (
    CONFIG_PATH,
    INFERENCE_DIR,
    #INFERENCE_INPUT_DIR,
    INFERENCE_OUTPUT_DIR,
    MODELS_ROOT,
    STACK_MODELS,
    parse_prediction_config,
)
#CONFIG_PATH = '/home/amismailov/mirna-repo/ml_pipeline/final_train_test_inference/inference/prediction_config.json'

INFERENCE_INPUT_DIR = Path('/home/amismailov/mirna-repo/ml_pipeline/data/RCC_python')
# processed inference input data: https://www.kaggle.com/datasets/ismailovaly/mirna-prediction-project
RCC_DIR = INFERENCE_INPUT_DIR
OUTPUT_DIR = INFERENCE_OUTPUT_DIR
FIG_PATH = INFERENCE_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RCC files:", sorted(p.name for p in RCC_DIR.glob("RCC_S*.csv")))
print("Config path:", CONFIG_PATH)
print("Models root:", MODELS_ROOT)
print("Stack models:", STACK_MODELS)
print("SingleCell module:", SingleCell.__module__)


RCC files: ['RCC_S1.csv', 'RCC_S2.csv', 'RCC_S3.csv', 'RCC_S4.csv', 'RCC_S5.csv']
Config path: /home/amismailov/mirna-repo/ml_pipeline/final_train_test_inference/inference/prediction_config.json
Models root: /home/amismailov/mirna-repo/ml_pipeline/final_train_test_inference/train/results
Stack models: ('tabpack', 'tabm', 'xgb_optuna')
SingleCell module: preprocessor


# 1-7: prediction step by step: data preparation, normalization, KNN imputing / KNN pseudobulk sampling, prediction
# 8: prediction using ine method, all above steps wrapped 

## 1. Config

- **K1** — single-cell (after KNN impute)
- **K2…K10** — pseudobulk KNN (without imputing)

Config schema (v2): `eligible_mirs`, `cohorts` = list of names, per-cohort maps `K1`…`K10` with `features` / metrics.
Use `parse_prediction_config()` (or `sc.target_info`) — there is no top-level `targets` key.


In [2]:
cfg = json.loads(CONFIG_PATH.read_text())
eligible, cohorts, target_info = parse_prediction_config(cfg)

print("Eligible miRNAs:", len(eligible))
print("Cohort sizes:", {k: len(v) for k, v in cohorts.items()})
print("Assignment rule:", str(cfg.get("assignment_rule", ""))[:140], "...")

example = "hsa-mir-100-5p"
info = target_info[example]
print(f"\nExample {example}:")
print("  assigned_cohort:", info["assigned_cohort"])
print("  n_features:", len(info["genes"]))
print("  m_bulk / m_assigned:", info.get("test_bulk"), "/", info.get("test_optimal_k"))


Eligible miRNAs: 159
Cohort sizes: {'K1': 9, 'K2': 21, 'K3': 11, 'K4': 11, 'K5': 27, 'K10': 80}
Assignment rule:  ...

Example hsa-mir-100-5p:
  assigned_cohort: K1
  n_features: 433
  m_bulk / m_assigned: 0.7941150855313355 / 0.9204779777557306


## 2. Load RCC dataset

Input: **cells × genes** CSV with columns `barcode`, `CellType` except for mRNA expression data (raw - counts)

In [3]:
SAMPLE = "RCC_S1"  # RCC_S1 … RCC_S5
INPUT_PATH = RCC_DIR / f"{SAMPLE}.csv"

# Для быстрой отладки: nrows=200. Для полного инференса: nrows=None
NROWS = 200

raw = pd.read_csv(INPUT_PATH, nrows=NROWS)
raw = raw.set_index("barcode")

print(raw.shape)
print(raw[["CellType"]].head())
print("ENSG columns:", sum(str(c).startswith("ENSG") for c in raw.columns))

(200, 21478)
                           CellType
barcode                            
AAACCCAGTAAGCAAT-1          T cells
AAACCCAGTCTGTGCG-1  Malignant cells
AAACCCAGTTAGAGAT-1          T cells
AAACCCATCACCTTGC-1         NK cells
AAACGAAAGCGATTCT-1          T cells
ENSG columns: 0


In [4]:
raw.head()

,CellType,AL627309.1,AL669831.5,FAM87B,LINC00115,FAM41C,AL645608.3,AL645608.1,SAMD11,NOC2L,...,MT-CYB,AC145212.1,MAFIP,AC011043.1,AL592183.1,AC007325.4,AC007325.2,AL354822.1,AC004556.1,AC240274.1
barcode,,,,,,,,,,,,,,,,,,,,,
AAACCCAGTAAGCAAT-1,T cells,0,0,0,0,0,0,0,0,0,...,9,0,0,0,0,0,0,0,0,0
AAACCCAGTCTGTGCG-1,Malignant cells,0,0,0,0,0,0,0,0,0,...,42,0,0,0,0,1,0,0,0,0
AAACCCAGTTAGAGAT-1,T cells,0,0,0,0,0,0,0,0,0,...,20,0,0,0,0,0,0,0,2,0
AAACCCATCACCTTGC-1,NK cells,0,0,0,0,0,0,0,0,0,...,16,0,0,0,0,0,0,0,0,0
AAACGAAAGCGATTCT-1,T cells,0,0,0,0,0,0,0,0,0,...,5,0,0,0,0,0,0,0,0,0


## 3. `SingleCell` and `StackPredictor` classes

`SingleCell` wraps preprocessing (TPM, KNN impute, KNN pseudobulk) and `StackPredictor`
(TabPack + DCNv2 + TabM → Ridge).


In [5]:
sc = SingleCell(
    device="cuda",  # or "cpu"
    preload_models=False,
)

print("Eligible miRNAs:", len(sc.available_mirnas))
print("K1 miRNAs:", len(sc.mirnas_for_cohort("K1")), sc.mirnas_for_cohort("K1"))
print("K2 miRNAs:", len(sc.mirnas_for_pseudobulk_k(2)))
print("Cohorts:", {k: len(v) for k, v in sc.cohorts.items()})


Eligible miRNAs: 159
K1 miRNAs: 9 ['hsa-let-7b-5p', 'hsa-mir-100-5p', 'hsa-mir-125b-5p', 'hsa-mir-142-3p', 'hsa-mir-19a-3p', 'hsa-mir-20a-5p', 'hsa-mir-21-5p', 'hsa-mir-27b-3p', 'hsa-mir-335-5p']
K2 miRNAs: 21
Cohorts: {'K1': 9, 'K2': 21, 'K3': 11, 'K4': 11, 'K5': 27, 'K10': 80}


## 4. Preprocessing step by step (single-cell K1)

### Step 4.1 — align genes

`prepare_input()` adjusts matrix to fix set of mRNA **17 392 ENSG**, non existing genes filled by 0.

In [6]:
counts_gc = sc.prepare_input(raw)
print("genes × cells:", counts_gc.shape)
print("gene order fixed:", counts_gc.index[:3].tolist())

genes × cells: (17392, 200)
gene order fixed: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419']


### Step 4.2 — TPM and log2 normalization

Normalization: counts → RPK → TPM → `log2(TPM + 1)`.

In [7]:
log_tpm_gc = sc.TPM(counts_gc, enforce_mrna_standard=False)
log_tpm_cells = log_tpm_gc.T  # cells × genes — формат для stack

print("log2(TPM+1) cells × genes:", log_tpm_cells.shape)
log_tpm_cells.iloc[:3, :3]

log2(TPM+1) cells × genes: (200, 17392)


,ENSG00000000003,ENSG00000000005,ENSG00000000419
barcode,,,
AAACCCAGTAAGCAAT-1,0.0,0.0,0.0
AAACCCAGTCTGTGCG-1,0.0,0.0,0.0
AAACCCAGTTAGAGAT-1,0.0,0.0,0.0


### Step 4.3 — KNN imputation (only for K1 single-cell)

Zeros in `log2(TPM+1)` replaces with average of k=5 nearest neighbors 

In [8]:
log_tpm_imputed_gc = sc.knn_impute_log_tpm(log_tpm_gc, knn_k=5)
log_tpm_imputed = log_tpm_imputed_gc.T

zeros_before = (log_tpm_cells == 0).sum().sum()
zeros_after = (log_tpm_imputed == 0).sum().sum()
print(f"zeros before/after impute: {zeros_before} → {zeros_after}")

zeros before/after impute: 2928994 → 1246511


### Step 4.4 — Stack prediction (K1 cohort)

For each miRNA: **TabPack + DCNv2 + TabM → Ridge stack**


In [9]:
# A: step by step (as above) + predict
pred_manual = sc.predict(log_tpm_imputed, mirnas=sc.mirnas_for_cohort("K1"))
print("manual K1 predictions:", pred_manual.shape)

# B: one-shot from raw counts (recommended for K1)
pred_k1 = sc.predict_single_cell_knn_imputed(raw)
print("one-shot K1 predictions:", pred_k1.shape)
pred_k1.iloc[:5, :5]

Stack prediction for 10 miRNAs...
Loading final_train stack models (tabpack + dcnv2 + tabm)...
✔ Stack ready: 152 eligible miRNAs
manual K1 predictions: (200, 10)
Detected gene symbols. Mapping to ENSG IDs...
Loading HGNC → ENSG mapping...
Replacing 21478 genes...
✔ Found ENSG for 21462/21478 genes (99.93%)
✔ After removing duplicates: 21401 unique ENSG
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 10 miRNAs...
one-shot K1 predictions: (200, 10)


,hsa-let-7b-5p,hsa-mir-100-5p,hsa-mir-125b-5p,hsa-mir-142-3p,hsa-mir-20a-5p
barcode,,,,,
AAACCCAGTAAGCAAT-1,13.394669,8.317229,12.695134,4.299815,14.050874
AAACCCAGTCTGTGCG-1,10.558126,11.890453,10.911486,1.623876,13.989228
AAACCCAGTTAGAGAT-1,13.135668,5.678976,12.879438,4.237981,13.955160
AAACCCATCACCTTGC-1,12.986422,10.043005,13.637645,4.984918,14.166948
AAACGAAAGCGATTCT-1,12.758871,9.771784,12.211661,4.289254,13.212701


## 5. Pseudobulk inference (K = 2, 3, 4, 5, 10)

For pseudobulk:
1. In PCA-space (log1p CPM, top HVG) find K nearest neighbors for each cell
2. Sum K cell raw counts → pseudobulk counts
3. TPM → stack **without** KNN impute

In [11]:
K_PB = 2 # example 
pred_pb_k2 = sc.predict_knn_pseudobulk(raw, K=K_PB)
print(f"PB K={K_PB}:", pred_pb_k2.shape)
print("miRNAs:", list(pred_pb_k2.columns[:5]), "...")
pred_pb_k2.iloc[:5, :5]

✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
PB K=2: (200, 18)
miRNAs: ['hsa-let-7d-5p', 'hsa-let-7i-5p', 'hsa-mir-1301-3p', 'hsa-mir-130a-3p', 'hsa-mir-135b-5p'] ...


,hsa-let-7d-5p,hsa-let-7i-5p,hsa-mir-1301-3p,hsa-mir-130a-3p,hsa-mir-135b-5p
barcode,,,,,
AAACCCAGTAAGCAAT-1,10.004285,8.111323,6.401043,3.461318,1.431046
AAACCCAGTCTGTGCG-1,7.444339,8.423554,6.246747,1.785949,6.389971
AAACCCAGTTAGAGAT-1,10.536684,10.997214,5.386059,5.009338,0.286757
AAACCCATCACCTTGC-1,10.563401,10.854077,5.408475,0.446886,0.000000
AAACGAAAGCGATTCT-1,10.899364,9.633400,4.905890,5.212992,0.000000


## 8. All in one method — `predict_all`

One method for all **eligible** miRNAs: K1 → single-cell + impute, K2…K10 → pseudobulk


In [10]:
SAMPLES = ["RCC_S1", "RCC_S2", "RCC_S3", "RCC_S4", "RCC_S5"]

for sample in SAMPLES:
    INPUT_PATH = RCC_DIR / f"{sample}.csv"
    raw_full = pd.read_csv(INPUT_PATH)
    raw_full = raw_full.set_index("barcode")

    pred_all = sc.predict_all(raw_full)
    out_path = OUTPUT_DIR / f"{sample}.csv"
    pred_all.to_csv(out_path)
    print(sample, pred_all.shape, "→", out_path)


Full inference: 7402 cells, 152 eligible miRNAs
  K1 single-cell + KNN impute: 10 miRNAs
Detected gene symbols. Mapping to ENSG IDs...
Loading HGNC → ENSG mapping...
Replacing 21478 genes...
✔ Found ENSG for 21462/21478 genes (99.93%)
✔ After removing duplicates: 21401 unique ENSG
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 10 miRNAs...
  K2 KNN pseudobulk (K=2): 18 miRNAs
Detected gene symbols. Mapping to ENSG IDs...
Loading HGNC → ENSG mapping...
Replacing 21477 genes...
✔ Found ENSG for 21462/21477 genes (99.93%)
✔ After removing duplicates: 21401 unique ENSG
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ 

In [36]:
import os
RCC_DIR = Path('/home/amismailov/mirna-repo/ml_pipeline/data/rcc_inference')
files = os.listdir(RCC_DIR)[18:]

for sample in files:
    INPUT_PATH = RCC_DIR / sample
    raw_full = pd.read_csv(INPUT_PATH)
    raw_full = raw_full.set_index("barcode")

    pred_all = sc.predict_all(raw_full)
    out_path = OUTPUT_DIR / sample
    pred_all.to_csv(out_path)
    print(sample, pred_all.shape, "→", out_path)


Full inference: 1125 cells, 152 eligible miRNAs
  K1 single-cell + KNN impute: 10 miRNAs
Detected gene symbols. Mapping to ENSG IDs...
Loading HGNC → ENSG mapping...
Replacing 14759 genes...
✔ Found ENSG for 14748/14759 genes (99.93%)
✔ After removing duplicates: 14741 unique ENSG
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 10 miRNAs...
  K2 KNN pseudobulk (K=2): 18 miRNAs
Detected gene symbols. Mapping to ENSG IDs...
Loading HGNC → ENSG mapping...
Replacing 14758 genes...
✔ Found ENSG for 14748/14758 genes (99.93%)
✔ After removing duplicates: 14741 unique ENSG
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ Found length for 17392/17392 genes (100.00%)
Stack prediction for 18 miRNAs...
✔ 

In [63]:
import os
RCC_DIR = Path('/home/amismailov/mirna-repo/ml_pipeline/data/RCC_h')
files = os.listdir(RCC_DIR)

for sample in files:
    INPUT_PATH = RCC_DIR / sample
    raw_full = pd.read_csv(INPUT_PATH)
    raw_full = raw_full.set_index("barcode")

    pred_all = sc.predict_all(raw_full)
    out_path = OUTPUT_DIR / 'RCC_h' / sample
    pred_all.to_csv(out_path)
    print(sample, pred_all.shape, "→", out_path)


Full inference: 1126 cells, 16115 genes, 152 eligible miRNAs
  K1 single-cell + KNN impute: 10 miRNAs
  K2 KNN pseudobulk (K=2): 18 miRNAs
  K3 KNN pseudobulk (K=3): 16 miRNAs
  K4 KNN pseudobulk (K=4): 19 miRNAs
  K5 KNN pseudobulk (K=5): 20 miRNAs
  K10 KNN pseudobulk (K=10): 69 miRNAs
✔ Done: 1126 cells × 152 miRNAs
p3_counts.csv (1126, 152) → /home/amismailov/mirna-repo/ml_pipeline/data/inference_outputs/RCC_h/p3_counts.csv
Full inference: 615 cells, 16243 genes, 152 eligible miRNAs
  K1 single-cell + KNN impute: 10 miRNAs
  K2 KNN pseudobulk (K=2): 18 miRNAs
  K3 KNN pseudobulk (K=3): 16 miRNAs
  K4 KNN pseudobulk (K=4): 19 miRNAs
  K5 KNN pseudobulk (K=5): 20 miRNAs
  K10 KNN pseudobulk (K=10): 69 miRNAs
✔ Done: 615 cells × 152 miRNAs
p14_counts.csv (615, 152) → /home/amismailov/mirna-repo/ml_pipeline/data/inference_outputs/RCC_h/p14_counts.csv
Full inference: 915 cells, 16246 genes, 152 eligible miRNAs
  K1 single-cell + KNN impute: 10 miRNAs
  K2 KNN pseudobulk (K=2): 18 miRNAs

In [9]:
import os
RCC_DIR = Path('/home/amismailov/mirna-repo/ml_pipeline/data/')

INPUT_PATH = RCC_DIR / 'RCC_huge_counts.csv'
raw_full = pd.read_csv(INPUT_PATH)
raw_full = raw_full.set_index("barcode")

pred_all = sc.predict_all(raw_full)
out_path = OUTPUT_DIR / 'RCC_huge_preds_v2.csv'
pred_all.to_csv(out_path)

Full inference: 106462 cells, 26838 genes, 159 eligible miRNAs
  K1 single-cell + KNN impute: 9 miRNAs
  K2 KNN pseudobulk (K=2): 21 miRNAs
  K3 KNN pseudobulk (K=3): 11 miRNAs
  K4 KNN pseudobulk (K=4): 11 miRNAs
  K5 KNN pseudobulk (K=5): 27 miRNAs
  K10 KNN pseudobulk (K=10): 80 miRNAs
✔ Done: 106462 cells × 159 miRNAs


NameError: name 'sample' is not defined

In [10]:
import os
RCC_DIR = Path('/home/amismailov/mirna-repo/ml_pipeline/data/')

INPUT_PATH = RCC_DIR / 'RCC_counts.csv'
raw_full = pd.read_csv(INPUT_PATH)
raw_full = raw_full.set_index("barcode")

pred_all = sc.predict_all(raw_full)
out_path = OUTPUT_DIR / 'RCC_cancer_huge_preds_v2.csv'
pred_all.to_csv(out_path)

Full inference: 43818 cells, 19152 genes, 159 eligible miRNAs
  K1 single-cell + KNN impute: 9 miRNAs
  K2 KNN pseudobulk (K=2): 21 miRNAs
  K3 KNN pseudobulk (K=3): 11 miRNAs
  K4 KNN pseudobulk (K=4): 11 miRNAs
  K5 KNN pseudobulk (K=5): 27 miRNAs
  K10 KNN pseudobulk (K=10): 80 miRNAs
✔ Done: 43818 cells × 159 miRNAs


In [61]:
import importlib
importlib.reload(preprocessor)
from preprocessor import SingleCell

sc = SingleCell(
    device="cuda",  # or "cpu"
    preload_models=False,
)